In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# ── Cell 0: load the finished data ──
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs, cartopy.feature as cfeature
import warnings; warnings.filterwarnings('ignore')

BASE = rf'{DATA_ROOT}/new_climate'; NAV_OUT = rf'{BASE}\siconc\_nav'; ARC_OUT = rf'{BASE}\wave\_arctic'; M5_OUT = rf'{BASE}\_M5'
MODELS = {'ACCESS': 'ACCESS-CM2', 'ECEARTH': 'EC-Earth3'}
TAGS = ['his', 'ssp126', 'ssp585']; SSPS = ['ssp126', 'ssp585']; SSP_LABEL = {'ssp126': 'SSP1-2.6', 'ssp585': 'SSP5-8.5'}
PASS = ['NEP (Northeast Passage)', 'NWP (Northwest Passage)', 'TSR (Transpolar Sea Route)', 'Fram Strait approach']
WMO = ['hs_lt1p25', 'hs_1p25_2p5', 'hs_2p5_4', 'hs_4_6', 'hs_6_9', 'hs_ge9']; WMO_LAB = ['<1.25 m', '1.25–2.5 m', '2.5–4 m', '4–6 m', '6–9 m', '≥9 m']

D = {}
for m in MODELS:
    z = np.load(os.path.join(M5_OUT, f'M5_masks_{m}.npz'), allow_pickle=True)
    d = {'VALID': z['VALID'], 'P_PASS': z['P_PASS'].astype(str), 'P_LAT': z['P_LAT'], 'P_LON': z['P_LON'], 'HIS_NAV': z['HIS_NAV'],
         'NEW': {'ssp126': z['NEW_126'], 'ssp585': z['NEW_585']}, 'FOD': {'ssp126': z['FOD_126'], 'ssp585': z['FOD_585']}}
    d['W'] = (3090.0 * np.cos(np.deg2rad(d['P_LAT']))).astype(np.float32)
    d['NAV'] = {t: {k: np.array(v) for k, v in np.load(os.path.join(NAV_OUT, f'nav_{m}_{t}.npz')).items()} for t in TAGS}
    d['WAV'] = {t: {k: np.array(v) for k, v in np.load(os.path.join(ARC_OUT, f'arctic_{m}_{t}.npz')).items()} for t in TAGS}
    for t in TAGS:
        for k in list(d['NAV'][t]):
            if d['NAV'][t][k].ndim == 2: d['NAV'][t][k][:, ~d['VALID']] = 0
        for k in ['nvalid'] + WMO: d['WAV'][t][k][:, ~d['VALID']] = 0
    D[m] = d
    print(m, '| VALID', d['VALID'].sum(), '| HIS_NAV', d['HIS_NAV'].sum(), '| NEW 126/585', d['NEW']['ssp126'].sum(), d['NEW']['ssp585'].sum())


In [ ]:
# ── Cell A: common settings ──
import matplotlib.path as mpath
DPI = 500
SAVE_DIR = None                                   # e.g. 'fig_panels'; None = show only, do not save
SSP_C = {'ssp126': '#3b7dd8', 'ssp585': '#e0453f'}
RTAG = dict(zip(PASS, ['R1', 'R2', 'R3', 'R4']))
# M2 Table 1 (A_it specification) full point estimates, significant or not; reference category = 1.0 (β=0)
IRR_HS = {'hs_lt1p25': 1.0, 'hs_1p25_2p5': 1.0036, 'hs_2p5_4': 1.0227, 'hs_4_6': 1.0462, 'hs_6_9': 1.0523, 'hs_ge9': 1.1989}
IRR_U  = {'u_lt8': 1.0, 'u_8_10p8': 1.0027, 'u_10p8_13p9': 1.0100, 'u_13p9_17p2': 1.0303, 'u_17p2_20p8': 1.0492, 'u_ge20p8': 1.1218}
HAZ_DAYS = {'wave': ['hs_2p5_4', 'hs_4_6', 'hs_6_9', 'hs_ge9'], 'wind': ['u_13p9_17p2', 'u_17p2_20p8', 'u_ge20p8']}
HAZ_NAME = {'wave': 'High-wave days (H$_s$ ≥ 2.5 m)', 'wind': 'High-wind days (U$_{10}$ ≥ 13.9 m s$^{-1}$)'}

WIND = {m: {t: {k: np.array(v) for k, v in np.load(os.path.join(ARC_OUT, f'arctic_wind_{m}_{t}.npz')).items()} for t in TAGS} for m in MODELS}
for m in MODELS:
    for t in TAGS:
        for k in [k for k in WIND[m][t] if WIND[m][t][k].ndim == 2]: WIND[m][t][k][:, ~D[m]['VALID']] = 0
SEA2 = pd.concat([pd.read_csv(os.path.join(M5_OUT, f'M5_seastate_{m}.csv')).assign(model=m) for m in MODELS], ignore_index=True)
GRP_NAME = {'Traditional': 'historically navigable', 'New': 'newly navigable (by 2081–2100)'}

def strip(ax):
    """Unlabelled version: remove all text (titles, axis labels, tick numbers, in-plot values, legend); keep ticks, axis lines and grid."""
    ax.set_title(''); ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticklabels([]); ax.set_yticklabels([])
    for t in list(ax.texts): t.remove()
    if ax.get_legend(): ax.get_legend().remove()

def finish(fig, name, clean):
    if SAVE_DIR: fig.savefig(os.path.join(SAVE_DIR, f'{name}_{"clean" if clean else "labeled"}.png'), dpi=DPI, bbox_inches='tight', transparent=clean)
    plt.show()


In [ ]:
# ── Cell B: first-opening decade maps, one figure = one model × one scenario ──
DEC_LAB = ['2020s', '2030s', '2040s', '2050s', '2060s', '2070s', '2080s', '2090s']
DEC_COLORS = ['#ffffb2', '#fed976', '#feb24c', '#fd8d3c', '#f03b20', '#bd0026', '#7a0177', '#49006a']
cmap_dec = ListedColormap(DEC_COLORS); norm_dec = BoundaryNorm(np.arange(-0.5, len(DEC_LAB) + 0.5, 1), cmap_dec.N)
ALREADY_C, NEVER_C, OCEAN_C, LAND_C = '#c7e0f0', '#fefdfd', '#e8f0f6', '#c9c9c9'
TAG_POS = {'NEP (Northeast Passage)': (74, 128), 'NWP (Northwest Passage)': (74, -100), 'TSR (Transpolar Sea Route)': (86, -100), 'Fram Strait approach': (74, -15)}
REG_LS = (0, (4, 2))
_t = np.linspace(0, 2 * np.pi, 300); CIRCLE = mpath.Path(np.vstack([np.sin(_t), np.cos(_t)]).T * 0.5 + 0.5)
LAT = np.arange(-90, 90.001, 0.5); LON = np.arange(0, 360, 0.5); L180 = ((LON + 180) % 360) - 180
ORD = np.argsort(L180); LONP = L180[ORD]; LONW = np.concatenate([LONP, [LONP[0] + 360]]); KI = LAT >= 59; LATP = LAT[KI]

def to_grid_m(m, v):
    d = D[m]; flat = np.round((d['P_LAT'] + 90) / .5).astype(int) * 720 + np.round((d['P_LON'] % 360) / .5).astype(int) % 720
    g = np.full(361 * 720, np.nan, np.float32); g[flat] = v; G = g.reshape(361, 720)[:, ORD][KI]; return np.concatenate([G, G[:, :1]], 1)

def fig_map(m, ssp, clean=False):
    fig = plt.figure(figsize=(3.6, 3.6), dpi=DPI)
    ax = plt.axes(projection=ccrs.LambertAzimuthalEqualArea(central_longitude=0, central_latitude=90))
    ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree()); ax.set_boundary(CIRCLE, transform=ax.transAxes)
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_C, zorder=0); ax.add_feature(cfeature.LAND, facecolor=LAND_C, edgecolor='none', zorder=3)
    ax.add_feature(cfeature.COASTLINE, lw=.3, edgecolor='#777', zorder=4)
    ax.gridlines(lw=.3, color='#8e9092', alpha=.7, zorder=5, xlocs=range(-180, 180, 45), ylocs=[70, 80])
    f = D[m]['FOD'][ssp]
    ax.pcolormesh(LONW, LATP, to_grid_m(m, np.where(f == 99, 1.0, np.nan)), cmap=ListedColormap([ALREADY_C]), vmin=0, vmax=1, transform=ccrs.PlateCarree(), shading='auto', zorder=1)
    ax.pcolormesh(LONW, LATP, to_grid_m(m, np.where(f == -1, 1.0, np.nan)), cmap=ListedColormap([NEVER_C]), vmin=0, vmax=1, transform=ccrs.PlateCarree(), shading='auto', zorder=1)
    fd = f.astype(np.float32); fd[(fd < 0) | (fd == 99)] = np.nan
    ax.pcolormesh(LONW, LATP, to_grid_m(m, fd), cmap=cmap_dec, norm=norm_dec, transform=ccrs.PlateCarree(), shading='auto', zorder=2)
    for latc in (66.5, 82.0): ax.plot(np.linspace(-180, 180, 361), np.full(361, latc), color='#111', lw=1.0, ls=REG_LS, transform=ccrs.PlateCarree(), zorder=8)
    for x in (15, -156, -45): ax.plot([x, x], [66.5, 82], color='#111', lw=1.0, ls=REG_LS, transform=ccrs.PlateCarree(), zorder=8)
    if not clean:
        for p, (la, lo) in TAG_POS.items():
            ax.text(lo, la, RTAG[p], transform=ccrs.PlateCarree(), fontsize=8, fontweight='bold', ha='center', va='center', zorder=9, path_effects=[pe.withStroke(linewidth=2.2, foreground='white')])
        ax.text(180, 61.5, '180°', transform=ccrs.PlateCarree(), fontsize=6, ha='center', va='bottom', color='#333', zorder=9)
        ax.text(0, 61.5, '0°', transform=ccrs.PlateCarree(), fontsize=6, ha='center', va='top', color='#333', zorder=9)
        ax.text(-100, 70.7, '70°N', transform=ccrs.PlateCarree(), fontsize=5.5, ha='center', va='bottom', color='#333', zorder=9)
        ax.text(-100, 80.7, '80°N', transform=ccrs.PlateCarree(), fontsize=5.5, ha='center', va='bottom', color='#333', zorder=9)
        ax.set_title(f'{MODELS[m]}  {SSP_LABEL[ssp]}', fontsize=9, pad=4)
    finish(fig, f'map_{m}_{ssp}', clean)

for m in MODELS:
    for ssp in SSPS:
        fig_map(m, ssp, clean=False); fig_map(m, ssp, clean=True)


In [ ]:
# ── Cell C: high-wave / high-wind bar charts, one figure = one hazard × one model; UNIT sets the y-axis definition ──
UNIT = 'relative'                                # 'total' = 10^6 km²·days/yr | 'percell' = days/yr per cell | 'share' = % of navigable days | 'relative' = ÷ traditional route 1985–2014 total exposure
PERS = ['2041–2060', '2061–2080', '2081–2100']
PER_RANGE = {'2041–2060': (2041, 2060), '2061–2080': (2061, 2080), '2081–2100': (2081, 2100)}
YLAB = {'total': 'Area-integrated exposure\n(10$^6$ km$^2$ · days yr$^{-1}$)', 'percell': 'Days per year (per cell)',
        'share': 'Share of navigable days (%)', 'relative': 'Exposure relative to\nhistorical traditional route (= 1)'}
BARS = [(g, s) for g in GRP_NAME for s in SSPS]
BW = 0.22                                        # bar width
PSTEP = 1.3                                      # spacing between period centres (total width of four bars ≈ 4·BW + 0.03, must be < PSTEP)
OFFS = np.array([-1.5, -0.5, 0.5, 1.5]) * (BW + 0.01)

def pan_mask(m, grp=None, ssp=None):
    d = D[m]; base = d['VALID'] & (d['P_LAT'] >= 66.5) & (d['P_PASS'] != '')
    if grp == 'Traditional': base &= d['HIS_NAV']
    elif grp == 'New': base &= d['NEW'][ssp]
    return base

def total_days(m, tag, cells, y0, y1, hz):
    """Area-integrated high-wave/high-wind navigable days (km²·days/yr)."""
    d = D[m]; Wv = d['WAV'][tag]; src = Wv if hz == 'wave' else WIND[m][tag]; ym = Wv['ym']
    k = (ym // 100 >= y0) & (ym // 100 <= y1)
    days = sum(src[b][k][:, cells].astype(np.float32) for b in HAZ_DAYS[hz]).sum(0) / (y1 - y0 + 1)
    return float(days @ d['W'][cells])

REF_TOT = {(hz, m): total_days(m, 'his', pan_mask(m, 'Traditional'), 1985, 2014, hz) for hz in HAZ_DAYS for m in MODELS}   # traditional route 1985–2014

def bar_val(m, ssp, per, grp, hz):
    d = D[m]; cells = pan_mask(m, grp, ssp); y0, y1 = PER_RANGE[per]
    tot = total_days(m, ssp, cells, y0, y1, hz)
    if UNIT == 'total':    return tot / 1e6
    if UNIT == 'relative': return tot / REF_TOT[(hz, m)]
    Wv = d['WAV'][ssp]; ym = Wv['ym']; k = (ym // 100 >= y0) & (ym // 100 <= y1)
    nval = Wv['nvalid'][k][:, cells].astype(np.float32).sum(0) / (y1 - y0 + 1); area = d['W'][cells]
    if UNIT == 'percell':  return tot / area.sum()
    return 100 * tot / float(nval @ area)                                                        # share

VALS = {(hz, m, ssp, per, grp): bar_val(m, ssp, per, grp, hz) for hz in HAZ_DAYS for m in MODELS for ssp in SSPS for per in PERS for grp in GRP_NAME}
BAR_YMAX = {hz: max(v for k, v in VALS.items() if k[0] == hz) * 1.15 for hz in HAZ_DAYS}                   # both models share the y-axis for a given hazard
print(f'[{UNIT}]'); print(pd.Series(VALS).unstack(level=3).round(3).to_string())
if UNIT == 'relative': print('Reference values (10^6 km²·days/yr):', {f'{hz} {m}': round(v / 1e6, 1) for (hz, m), v in REF_TOT.items()})

def fig_bar(hz, m, clean=False):
    fig, ax = plt.subplots(figsize=(3.8, 2.6), dpi=DPI)
    for i, per in enumerate(PERS):
        xc = i * PSTEP
        for (grp, ssp), off in zip(BARS, OFFS):
            v = VALS[(hz, m, ssp, per, grp)]
            ax.bar(xc + off, v, width=BW, color=SSP_C[ssp], edgecolor='white' if grp == 'New' else SSP_C[ssp],
                   hatch='///' if grp == 'New' else None, lw=.5, zorder=3)
            if not clean:
                ax.text(xc + off, v + BAR_YMAX[hz] * .01, f'{v:.2f}' if UNIT in ('share', 'relative') else f'{v:.0f}',
                        ha='center', va='bottom', fontsize=5.0, color='#333')
    if UNIT == 'relative': ax.axhline(1, color='#333', lw=.7, ls=':', zorder=2)
    ax.set_xticks([i * PSTEP for i in range(len(PERS))]); ax.set_xticklabels(PERS, fontsize=7.5)
    ax.set_xlim(-0.55, (len(PERS) - 1) * PSTEP + 0.55); ax.set_ylim(0, BAR_YMAX[hz])
    ax.tick_params(axis='y', labelsize=7); ax.grid(axis='y', lw=.3, color='#ddd', zorder=0)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    ax.set_title(f'{HAZ_NAME[hz]}  ·  {MODELS[m]}', fontsize=8); ax.set_ylabel(YLAB[UNIT], fontsize=7.5)
    if clean: strip(ax)
    finish(fig, f'bar_{hz}_{m}_{UNIT}', clean)

for hz in HAZ_DAYS:
    for m in MODELS:
        fig_bar(hz, m, clean=False); fig_bar(hz, m, clean=True)


In [ ]:
STACK = UNIT in ('total', 'relative')            # stacking is valid only for additive quantities
BW = 0.32 if STACK else 0.22
PSTEP = 1.0 if STACK else 1.3
OFFS = (np.array([-0.5, 0.5]) * (BW + 0.03)) if STACK else (np.array([-1.5, -0.5, 0.5, 1.5]) * (BW + 0.01))
BAR_YMAX = {hz: (max(VALS[(hz, m, s, per, 'Traditional')] + VALS[(hz, m, s, per, 'New')] for m in MODELS for s in SSPS for per in PERS)
                 if STACK else max(v for k, v in VALS.items() if k[0] == hz)) * 1.15 for hz in HAZ_DAYS}

def fig_bar(hz, m, clean=False):
    fig, ax = plt.subplots(figsize=(3.8, 2.6), dpi=DPI)
    fmt = (lambda v: f'{v:.2f}') if UNIT in ('share', 'relative') else (lambda v: f'{v:.0f}')
    for i, per in enumerate(PERS):
        xc = i * PSTEP
        if STACK:
            for ssp, off in zip(SSPS, OFFS):
                t = VALS[(hz, m, ssp, per, 'Traditional')]; n = VALS[(hz, m, ssp, per, 'New')]
                ax.bar(xc + off, t, width=BW, color=SSP_C[ssp], edgecolor=SSP_C[ssp], lw=.5, zorder=3)
                ax.bar(xc + off, n, bottom=t, width=BW, color=SSP_C[ssp], edgecolor='white', hatch='///', lw=.5, zorder=3)
                if not clean:
                    ax.text(xc + off, t + n + BAR_YMAX[hz] * .01, fmt(t + n), ha='center', va='bottom', fontsize=5.2, color='#333')
                    if t > BAR_YMAX[hz] * .08: ax.text(xc + off, t / 2, fmt(t), ha='center', va='center', fontsize=4.6, color='white')
                    if n > BAR_YMAX[hz] * .08: ax.text(xc + off, t + n / 2, fmt(n), ha='center', va='center', fontsize=4.6, color='white')
        else:
            for (grp, ssp), off in zip(BARS, OFFS):
                v = VALS[(hz, m, ssp, per, grp)]
                ax.bar(xc + off, v, width=BW, color=SSP_C[ssp], edgecolor='white' if grp == 'New' else SSP_C[ssp],
                       hatch='///' if grp == 'New' else None, lw=.5, zorder=3)
                if not clean: ax.text(xc + off, v + BAR_YMAX[hz] * .01, fmt(v), ha='center', va='bottom', fontsize=5.0, color='#333')
    if UNIT == 'relative': ax.axhline(1, color='#333', lw=.7, ls=':', zorder=2)
    ax.set_xticks([i * PSTEP for i in range(len(PERS))]); ax.set_xticklabels(PERS, fontsize=7.5)
    ax.set_xlim(-0.55, (len(PERS) - 1) * PSTEP + 0.55); ax.set_ylim(0, BAR_YMAX[hz])
    ax.tick_params(axis='y', labelsize=7); ax.grid(axis='y', lw=.3, color='#ddd', zorder=0)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    ax.set_title(f'{HAZ_NAME[hz]}  ·  {MODELS[m]}', fontsize=8); ax.set_ylabel(YLAB[UNIT], fontsize=7.5)
    if clean: strip(ax)
    finish(fig, f'bar_{hz}_{m}_{UNIT}{"_stack" if STACK else ""}', clean)


In [ ]:
# ── Cell D: hazard index lines, one figure = one hazard × one model; traditional solid, new dashed, 10-yr moving average ──
ROLL, MIN_WAVE_DAYS = 10, 10
LS = {'Traditional': '-', 'New': '--'}

def yearly_index(m, ssp, cells, hz):
    Wv = D[m]['WAV'][ssp]; src, irr = (Wv, IRR_HS) if hz == 'wave' else (WIND[m][ssp], IRR_U)
    nv = Wv['nvalid'][:, cells].astype(np.float32); area = D[m]['W'][cells]
    logm = sum(np.log(v) * src[b][:, cells].astype(np.float32) for b, v in irr.items())
    w = nv * area; num_m = (w * np.exp(logm)).sum(1); den_m = w.sum(1)
    yrs = Wv['ym'] // 100; Y = np.unique(yrs)
    num = np.array([num_m[yrs == y].sum() for y in Y]); den = np.array([den_m[yrs == y].sum() for y in Y])
    idx = num / den; idx[den / area.sum() < MIN_WAVE_DAYS] = np.nan
    return Y, pd.Series(idx).rolling(ROLL, center=True, min_periods=max(3, ROLL // 2)).mean().values

LINES = {}
for hz in HAZ_DAYS:
    for m in MODELS:
        d = D[m]; pan = d['VALID'] & (d['P_LAT'] >= 66.5) & (d['P_PASS'] != '')
        for ssp in SSPS:
            for grp, mask in [('Traditional', d['HIS_NAV']), ('New', d['NEW'][ssp])]:
                LINES[(hz, m, ssp, grp)] = yearly_index(m, ssp, pan & mask, hz)
LINE_YMAX = {hz: np.nanmax([v[1] for k, v in LINES.items() if k[0] == hz]) * 1.03 for hz in HAZ_DAYS}      # shared by both models for a given hazard

def fig_line(hz, m, clean=False):
    fig, ax = plt.subplots(figsize=(3.6, 2.6), dpi=DPI)
    for ssp in SSPS:
        for grp in LS:
            Y, v = LINES[(hz, m, ssp, grp)]; ax.plot(Y, v, color=SSP_C[ssp], ls=LS[grp], lw=1.5, zorder=3)
    ax.axhline(1, color='#333', lw=.7, ls=':')
    ax.set_xlim(2040, 2100); ax.set_xticks([2040, 2060, 2080, 2100]); ax.set_ylim(0.98, LINE_YMAX[hz])
    ax.tick_params(labelsize=7.5); ax.grid(axis='y', lw=.3, color='#ddd')
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    ax.set_title(f'Hazard index ({hz} weights)  ·  {MODELS[m]}', fontsize=8); ax.set_ylabel('Hazard-weighted index', fontsize=7.5)
    if clean: strip(ax)
    finish(fig, f'line_{hz}_{m}', clean)

for hz in HAZ_DAYS:
    for m in MODELS:
        fig_line(hz, m, clean=False); fig_line(hz, m, clean=True)
